# 02 — Model comparison (research only)

Evaluates the preregistered model families against the baselines on the **frozen** expanding-season
folds. Writes `artifacts/model_comparison.json` and **nothing the live page reads**.

**Two tracks, kept apart (A2.3):**

* **M1–M3 — independent structural.** `market_line` is not in their features.
* **M5 — market-anchored residual.** `residual = wins − market_line`; prediction adds it back. It
  **cannot** show this project beating a sportsbook or a market; the only claim it supports is
  whether structural information improved MAE relative to the archived consensus.

M4 (schedule Monte Carlo) is notebook `03`.

**Scope under `GO-TIER-B`.** §4 metrics 1–2 (MAE, ΔMAE) are computed. **Metrics 3–4 — the
side-taking hit rate and the price-implied break-even — are NOT computed**: both require gate C,
which is shut, and the quote columns are physically absent from the panel.

**Every number is reported twice** (A1.4): the headline fold set and the strictly-dated sensitivity.
The sensitivity is underpowered and can never be the headline; if the two disagree in sign, the
strict subset governs and the disagreement is the finding.

**One-shot (§7).** The gates fire once on the frozen folds and the result is written to JSON.
Re-running to obtain a different number is forbidden.

```bash
papermill futures/season_team_totals/02_model_comparison.ipynb /tmp/out.ipynb
```

## Section 1 — Parameters

Paths and seeds only. Folds, tier, eligibility and the M5 contract all come from the frozen
artifacts — nothing about the evaluation design is a parameter.

In [1]:
AUDIT_PATH   = None     # None -> futures/artifacts/data_audit.json
PANEL_PATH   = None     # None -> futures/data/team_season_panel.parquet
META_PATH    = None     # None -> futures/artifacts/dataset_metadata.json
OUT_PATH     = None     # None -> futures/artifacts/model_comparison.json
N_BOOTSTRAP  = 10000    # §7: season-block bootstrap resamples
SEED         = 20260802
WRITE_ARTIFACTS = True
RUN_TESTS    = True

### Interpreting the output

Silent by design. Note what is *absent*: no fold list, no model list, no threshold. Those are frozen
upstream and read, so this notebook cannot quietly redefine the experiment.

### What these tests guard

That the evaluation design cannot be injected as a parameter, and that the bootstrap is large enough
for the §7 interval to be stable.

In [2]:
if RUN_TESTS:
    assert isinstance(SEED, int) and N_BOOTSTRAP >= 10000, "§7 fixes 10,000 season-block resamples"
    for _n in ("FOLDS", "TEST_SEASONS", "MODELS", "ALPHA_GRID", "GATE_A_THRESHOLD"):
        assert _n not in dir(), f"{_n} must come from the frozen artifacts, never a parameter"
    print(f"✓ Section 1 tests passed | paths+seed only; {N_BOOTSTRAP:,} bootstrap resamples")

✓ Section 1 tests passed | paths+seed only; 10,000 bootstrap resamples


### Reading the test result

Confirms the parameter surface is paths, a seed and the resample count. Does **not** prove the
artifacts exist — Section 2.

## Section 2 — Gate, load, and hash-verify

Refuses to run unless the audit says `GO`/`GO-TIER-B`, then loads the panel and asserts it is the
one the metadata describes. Reads the frozen folds, the eligibility tables, the M5 contract and the
Tier-C lock.

In [3]:
import hashlib
import json
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start}")


REPO    = _find_repo_root(Path.cwd())
FUTURES = REPO / "futures"
ART_DIR = FUTURES / "artifacts"
AUDIT = Path(AUDIT_PATH) if AUDIT_PATH else ART_DIR / "data_audit.json"
PANEL = Path(PANEL_PATH) if PANEL_PATH else FUTURES / "data" / "team_season_panel.parquet"
META  = Path(META_PATH) if META_PATH else ART_DIR / "dataset_metadata.json"
OUT   = Path(OUT_PATH) if OUT_PATH else ART_DIR / "model_comparison.json"
AUDIT, PANEL, META, OUT = (p if p.is_absolute() else REPO / p for p in (AUDIT, PANEL, META, OUT))


def _rel(p) -> str:
    p = Path(p)
    try:
        return p.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(p.resolve())


def sha256_frame(df: pd.DataFrame) -> str:
    return hashlib.sha256(
        pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).values.tobytes()
    ).hexdigest()


audit = json.loads(AUDIT.read_text(encoding="utf-8"))
meta = json.loads(META.read_text(encoding="utf-8"))
VERDICT = audit["verdict"]
if not VERDICT.startswith("GO"):
    raise RuntimeError(f"audit verdict is {VERDICT} — 02 must not run")

TIER_C_OPEN = bool(audit.get("tier_c_open", False))
FOLDS        = list(audit["folds"]["test_seasons"])
FOLDS_STRICT = list(audit["folds_strict_sensitivity"]["test_seasons"])
TARGET       = audit["target"]["column"]
FEATURES     = list(meta["features"]["columns"])
ELIG         = {e["test_season"]: e for e in meta["eligibility"]["headline"]}
M5           = meta["m5_contract"]
M5_ELIG      = {e["test_season"]: e for e in M5["training_by_fold"]}

panel = pd.read_parquet(PANEL)
PANEL_HASH = sha256_frame(panel[meta["panel"]["columns"]])

sys.path.insert(0, str(FUTURES / "season_team_totals"))
from tier_lock import TierCViolation, assert_no_tier_c

ALLOWED_LITERALS = {VERDICT, audit.get("tier_available", ""),
                    "prior_off_epa_play", "prior_def_epa_play"} | set(
    panel["market_source"].dropna().astype(str).unique())


def guard(obj, where):
    assert_no_tier_c(obj, where, allowed_literals=ALLOWED_LITERALS, tier_c_open=TIER_C_OPEN)


RUN_AT = datetime.now(timezone.utc)
PROVENANCE = {"notebook": "futures/season_team_totals/02_model_comparison.ipynb",
              "run_at_utc": RUN_AT.isoformat(), "python": sys.version.split()[0],
              "platform": platform.platform(), "pandas": pd.__version__,
              "numpy": np.__version__, "seed": SEED, "audit_verdict": VERDICT}

print(f"verdict        : {VERDICT}  (gate C open: {TIER_C_OPEN})")
print(f"panel          : {len(panel):,} rows, {len(FEATURES)} structural features")
print(f"panel hash     : {PANEL_HASH[:16]}…  matches metadata: {PANEL_HASH == meta['panel']['frame_sha256']}")
print(f"headline folds : {FOLDS}")
print(f"strict folds   : {FOLDS_STRICT}")
print(f"M5 alpha grid  : {M5['alpha_grid']}  fallback {M5['fallback_alpha']}")

verdict        : GO-TIER-B  (gate C open: False)
panel          : 800 rows, 25 structural features
panel hash     : 039f0e0d9e5bc29f…  matches metadata: True
headline folds : [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024, 2025]
strict folds   : [2015, 2016, 2017, 2018]
M5 alpha grid  : [0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0]  fallback 10.0


### Interpreting the output

`GO-TIER-B`, gate C shut, 800 panel rows, **25** structural features, and a panel hash matching the
metadata — so this is the dataset `01` blessed, not a re-derivation.

10 headline folds and 4 strict folds are read, not chosen.

### What these tests guard

The gate is real, the panel is the audited one, `market_line` is **not** among the structural
features (A2.3), and the Tier-C guard is live before any modelling begins.

In [4]:
if RUN_TESTS:
    assert VERDICT in ("GO", "GO-TIER-B") and TIER_C_OPEN is False
    assert PANEL_HASH == meta["panel"]["frame_sha256"], "panel differs from the one 01 wrote"
    assert len(FEATURES) == 25 and "market_line" not in FEATURES, \
        "market_line must never be a structural feature (A2.3)"
    assert set(FOLDS_STRICT) <= set(FOLDS) and FOLDS == sorted(FOLDS)
    assert TARGET == "wins_half_ties"
    assert not ({"price_over", "price_under"} & set(panel.columns)), "quote columns leaked in"
    try:
        guard({"note": "a betting edge"}, "gate-selftest")
        raise AssertionError("the Tier-C guard is not active")
    except TierCViolation:
        pass
    print(f"✓ Section 2 tests passed | {VERDICT}, panel hash matches, {len(FEATURES)} features, "
          f"market_line excluded, guard live")

✓ Section 2 tests passed | GO-TIER-B, panel hash matches, 25 features, market_line excluded, guard live


### Reading the test result

The inputs are the audited ones and the fence is up. Does **not** prove the folds are *adequate* —
§3 already flagged the strict subset as underpowered.

## Section 3 — Fold machinery

For test season *T*: **structural models train on every settled season < T** (line-covered or not —
2015 trains on 2002–2014); **M5 trains only on line-covered seasons < T** (A2.2); **both evaluate on
the line-covered rows of T**. The eligibility tables from `01` are the authority, and this section
re-derives them and asserts agreement rather than trusting them.

In [5]:
def train_rows(T, line_covered_only=False):
    m = (panel["season"] < T) & panel["has_target"]
    if line_covered_only:
        m &= panel["line_covered"]
    return panel[m]


def eval_rows(T):
    return panel[(panel["season"] == T) & panel["line_covered"] & panel["has_target"]]


def inner_folds(train_seasons):
    # expanding-season inside the outer training window: validate on S_i, train on S_1..S_{i-1}
    s = sorted(train_seasons)
    return [(s[:i], s[i]) for i in range(1, len(s))]


fold_table = []
for T in FOLDS:
    tr, trm, ev = train_rows(T), train_rows(T, True), eval_rows(T)
    fold_table.append({
        "test_season": int(T),
        "n_train_structural": len(tr), "n_train_m5": len(trm), "n_eval": len(ev),
        "train_seasons_structural": sorted(int(s) for s in tr["season"].unique()),
        "train_seasons_m5": sorted(int(s) for s in trm["season"].unique()),
        "inner_folds_structural": len(inner_folds(tr["season"].unique())),
        "inner_folds_m5": len(inner_folds(trm["season"].unique())),
        "strict": bool(T in FOLDS_STRICT),
    })
ft = pd.DataFrame(fold_table)
print(ft[["test_season", "strict", "n_train_structural", "n_train_m5", "n_eval",
          "inner_folds_structural", "inner_folds_m5"]].to_string(index=False))

 test_season  strict  n_train_structural  n_train_m5  n_eval  inner_folds_structural  inner_folds_m5
        2015    True                 416          32      32                      12               0
        2016    True                 448          64      32                      13               1
        2017    True                 480          96      32                      14               2
        2018    True                 512         128      32                      15               3
        2019   False                 544         160      32                      16               4
        2020   False                 576         192      32                      17               5
        2021   False                 608         224      32                      18               6
        2022   False                 640         256      32                      19               7
        2024   False                 704         288      32                      21       

### Interpreting the output

Structural training grows 416 → 736 rows; M5 training grows 32 → 288. The gap is the whole point of
the left join in `01`: seasons without a market line still teach the structural models.

`inner_folds_m5` is **0 for 2015** — one training season cannot be split — which is exactly the
condition A2.2 predeclared the fallback alpha for.

### What these tests guard

No training row is from season ≥ T; structural training is strictly larger than the M5 subset;
evaluation is 32 line-covered rows per fold; and the re-derived eligibility **matches `01`'s
recorded tables exactly**. The splitter is also shown able to fail — a deliberately leaky splitter
that includes season T must be caught.

In [6]:
if RUN_TESTS:
    for r in fold_table:
        T = r["test_season"]
        assert max(r["train_seasons_structural"]) < T and max(r["train_seasons_m5"]) < T
        assert r["n_train_structural"] > r["n_train_m5"], "the market must not gate structural training"
        assert r["n_eval"] == 32
        assert r["train_seasons_structural"] == ELIG[T]["train_seasons"], "disagrees with 01"
        assert r["n_train_structural"] == ELIG[T]["n_train"]
        assert r["train_seasons_m5"] == M5_ELIG[T]["train_seasons"], "disagrees with the M5 contract"
        assert r["n_train_m5"] == M5_ELIG[T]["n_train"]
        assert (r["inner_folds_m5"] < 2) == M5_ELIG[T]["takes_fallback_alpha"]
    # RED CONTROL: a splitter that leaks the test season must be detectable
    _leaky = panel[(panel["season"] <= FOLDS[0]) & panel["has_target"]]
    assert _leaky["season"].max() == FOLDS[0], "the red control did not actually leak"
    assert train_rows(FOLDS[0])["season"].max() < FOLDS[0], "the real splitter leaks"
    print(f"✓ Section 3 tests passed | {len(FOLDS)} folds, structural {ft.n_train_structural.min()}–"
          f"{ft.n_train_structural.max()} rows vs M5 {ft.n_train_m5.min()}–{ft.n_train_m5.max()}, "
          f"32 eval rows each, eligibility matches 01, red control detected")

✓ Section 3 tests passed | 10 folds, structural 416–736 rows vs M5 32–320, 32 eval rows each, eligibility matches 01, red control detected


### Reading the test result

The splitter agrees with `01` fold for fold and cannot see the test season. Does **not** prove the
*models* respect it — that is enforced by fitting inside these row sets, checked next.

## Section 4 — Baselines B0–B3 (§4)

* **B0 — the archived market consensus.** Predict `market_line`. The benchmark that matters.
* **B1 — persistence.** Prior wins rescaled for a 16→17 game change.
* **B2 — league mean.** `games_scheduled / 2`.
* **B3 — shrunk persistence.** Prior win% shrunk toward .500 by a factor **fitted on training rows
  only**, closed-form least squares — no tuning knob, no peeking.

In [7]:
def fit_b3_shrinkage(tr):
    # least-squares k in: wins = games * (0.5 + k * (prior_win_pct - 0.5)); training rows only
    d = tr.dropna(subset=["prior_win_pct", TARGET])
    x = (d["prior_win_pct"] - 0.5) * d["games_scheduled"]
    y = d[TARGET] - 0.5 * d["games_scheduled"]
    return float((x * y).sum() / (x * x).sum())


def baselines_for(T):
    tr, ev = train_rows(T), eval_rows(T)
    k = fit_b3_shrinkage(tr)
    b1 = ev["prior_wins"] * (ev["games_scheduled"] / ev["prior_games"])
    return pd.DataFrame({
        "B0_market": ev["market_line"].to_numpy(float),
        "B1_persistence": b1.to_numpy(float),
        "B2_league_mean": (ev["games_scheduled"] / 2.0).to_numpy(float),
        "B3_shrunk": (ev["games_scheduled"] * (0.5 + k * (ev["prior_win_pct"] - 0.5))).to_numpy(float),
    }, index=ev.index), k


b3_k = {}
base_preds = {}
for T in FOLDS:
    base_preds[T], b3_k[T] = baselines_for(T)
print("B3 shrinkage k fitted per fold (training rows only):")
print("  " + "  ".join(f"{T}:{b3_k[T]:.3f}" for T in FOLDS))
print()
_b = pd.DataFrame({T: base_preds[T].isna().sum() for T in FOLDS}).T
print("null predictions by fold (must be zero):")
print(_b.sum().to_frame("nulls").T.to_string())

B3 shrinkage k fitted per fold (training rows only):
  2015:0.311  2016:0.315  2017:0.313  2018:0.310  2019:0.310  2020:0.317  2021:0.328  2022:0.340  2024:0.336  2025:0.340

null predictions by fold (must be zero):
       B0_market  B1_persistence  B2_league_mean  B3_shrunk
nulls          0               0               0          0


### Interpreting the output

The fitted shrinkage sits around **0.30–0.35** every fold — prior-season record is worth roughly a
third of its face value, which is why B2 beat raw persistence in `00`'s in-sample look. That the
estimate is stable across folds is reassuring: it is not chasing noise.

No baseline produces a null on any evaluation row.

### What these tests guard

Every baseline is computable on all 320 evaluation rows, B0 equals the archived line exactly, and
**B3's shrinkage never sees a test row** — refitting it on training-plus-test must change the value,
proving the fit is genuinely training-only.

In [8]:
if RUN_TESTS:
    for T in FOLDS:
        ev = eval_rows(T)
        assert base_preds[T].notna().all().all(), f"{T}: null baseline prediction"
        assert np.allclose(base_preds[T]["B0_market"], ev["market_line"]), "B0 must equal the line"
        assert np.allclose(base_preds[T]["B2_league_mean"] * 2, ev["games_scheduled"])
        assert 0.0 < b3_k[T] < 1.0, f"{T}: implausible shrinkage {b3_k[T]}"
    # B3's k is training-only: adding the test season must move it
    _T = FOLDS[0]
    _contaminated = fit_b3_shrinkage(panel[(panel["season"] <= _T) & panel["has_target"]])
    assert abs(_contaminated - b3_k[_T]) > 1e-9, "B3 shrinkage appears not to be training-only"
    print(f"✓ Section 4 tests passed | 4 baselines on {sum(len(base_preds[T]) for T in FOLDS)} rows, "
          f"B0 == the line, B3 k in [{min(b3_k.values()):.3f}, {max(b3_k.values()):.3f}] fitted "
          f"training-only")

✓ Section 4 tests passed | 4 baselines on 320 rows, B0 == the line, B3 k in [0.310, 0.340] fitted training-only


### Reading the test result

The baselines are complete and B3's one fitted quantity is provably training-only. Does **not** say
which baseline is hardest to beat — that is the metric table.

## Section 5 — M1: shrunk persistence + prior point differential (linear)

The simplest structural model §6 declares: ordinary least squares on prior win% and prior point
differential, predicting wins, refit per fold on training rows only. No hyperparameters, so nothing
to select — it is the honest floor for "does structure add anything to persistence".

In [9]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

M1_FEATURES = ["prior_win_pct", "prior_point_diff"]


def make_linear():
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", LinearRegression())])


m1_preds = {}
for T in FOLDS:
    tr, ev = train_rows(T), eval_rows(T)
    p = make_linear().fit(tr[M1_FEATURES], tr[TARGET])
    m1_preds[T] = pd.Series(p.predict(ev[M1_FEATURES]), index=ev.index)

print(f"M1 features: {M1_FEATURES}")
print(f"folds fitted: {len(m1_preds)}   predictions: {sum(len(v) for v in m1_preds.values())}")
print("prediction range by fold:")
print("  " + "  ".join(f"{T}:[{m1_preds[T].min():.1f},{m1_preds[T].max():.1f}]" for T in FOLDS[:5]) + " …")

M1 features: ['prior_win_pct', 'prior_point_diff']
folds fitted: 10   predictions: 320
prediction range by fold:
  2015:[5.8,9.7]  2016:[6.3,9.9]  2017:[6.1,10.0]  2018:[6.4,9.9]  2019:[5.7,9.6] …


### Interpreting the output

Predictions land in a plausible win range every fold. M1 is deliberately weak — two features, no
tuning — so it isolates how much of any later gain comes from the extra features rather than from
the modelling machinery.

### What these tests guard

Imputation and scaling are fitted **inside** the fold (a pipeline, not a pre-transformed frame), the
prediction count matches the evaluation rows, and predictions stay inside the achievable win range.

In [10]:
if RUN_TESTS:
    for T in FOLDS:
        ev = eval_rows(T)
        assert len(m1_preds[T]) == len(ev) and m1_preds[T].notna().all()
        assert m1_preds[T].between(-2, 20).all(), f"{T}: implausible M1 prediction"
    # the imputer must be fold-fitted: a pipeline refit on a different window gives different output
    _T = FOLDS[-1]
    _alt = make_linear().fit(train_rows(FOLDS[0])[M1_FEATURES], train_rows(FOLDS[0])[TARGET])
    _alt_pred = _alt.predict(eval_rows(_T)[M1_FEATURES])
    assert not np.allclose(_alt_pred, m1_preds[_T]), "M1 appears not to be refit per fold"
    print(f"✓ Section 5 tests passed | M1 fitted on {len(FOLDS)} folds, "
          f"{sum(len(v) for v in m1_preds.values())} predictions, refit-per-fold verified")

✓ Section 5 tests passed | M1 fitted on 10 folds, 320 predictions, refit-per-fold verified


### Reading the test result

M1 is refit every fold — a model fitted on the earliest window gives different answers, which is
what the check demonstrates. Does **not** yet say whether M1 is any good.

## Section 6 — M2: Ridge on the full structural feature set

All 25 features, median-imputed and standardized **inside each fold**, with `alpha` chosen by
**inner expanding-season validation** within the training window — never on the test season.

The alpha grid is frozen here, before first execution, and is the same grid A2.2 fixed for M5.
Widening it after seeing a fold result is forbidden by the rule that governs §7.

In [11]:
ALPHA_GRID = tuple(M5["alpha_grid"])          # frozen; shared with the M5 contract
FALLBACK_ALPHA = float(M5["fallback_alpha"])


def make_ridge(alpha):
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", Ridge(alpha=alpha, random_state=None))])


def select_alpha(tr, features, target_col):
    # inner expanding-season selection; smallest alpha among ties; fallback when <2 inner folds
    folds = inner_folds(tr["season"].unique())
    if len(folds) < 2:
        return FALLBACK_ALPHA, True, {}
    scores = {}
    for a in ALPHA_GRID:
        errs = []
        for tr_seasons, va_season in folds:
            i_tr = tr[tr["season"].isin(tr_seasons)]
            i_va = tr[tr["season"] == va_season]
            p = make_ridge(a).fit(i_tr[features], i_tr[target_col])
            errs.append(np.abs(p.predict(i_va[features]) - i_va[target_col]).mean())
        scores[a] = float(np.mean(errs))
    best = min(scores, key=lambda a: (scores[a], a))
    return float(best), False, scores


m2_preds, m2_alpha = {}, {}
for T in FOLDS:
    tr, ev = train_rows(T), eval_rows(T)
    a, fb, _ = select_alpha(tr, FEATURES, TARGET)
    m2_alpha[T] = {"alpha": a, "fallback": fb}
    m2_preds[T] = pd.Series(make_ridge(a).fit(tr[FEATURES], tr[TARGET]).predict(ev[FEATURES]),
                            index=ev.index)

print(f"alpha grid (frozen): {ALPHA_GRID}")
print("selected alpha by fold (inner expanding-season):")
print("  " + "  ".join(f"{T}:{m2_alpha[T]['alpha']:g}" for T in FOLDS))
print(f"folds taking the fallback: {[T for T in FOLDS if m2_alpha[T]['fallback']]}")

alpha grid (frozen): (0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0)
selected alpha by fold (inner expanding-season):
  2015:100  2016:300  2017:300  2018:100  2019:100  2020:100  2021:100  2022:100  2024:100  2025:100
folds taking the fallback: []


### Interpreting the output

Selected alphas are large — heavy regularization — which is what you expect with 25 features, a few
hundred training rows, and a target this noisy. No structural fold takes the fallback: every one has
at least 12 inner validation seasons.

Alpha varying by fold is not instability; it is the selection doing its job on a growing window.

### What these tests guard

The selected alpha is always **in the frozen grid**, selection uses only training seasons, and the
inner loop never touches the test season. The selector is also shown to *discriminate* — a degenerate
selector returning a constant would pass everything else here.

In [12]:
if RUN_TESTS:
    for T in FOLDS:
        assert m2_alpha[T]["alpha"] in ALPHA_GRID or m2_alpha[T]["fallback"]
        assert len(m2_preds[T]) == len(eval_rows(T)) and m2_preds[T].notna().all()
        assert m2_preds[T].between(-2, 20).all()
        # inner folds are strictly inside the training window
        for tr_s, va_s in inner_folds(train_rows(T)["season"].unique()):
            assert va_s < T and max(tr_s) < va_s
    # the selector must be able to prefer different alphas on different data
    _s1 = select_alpha(train_rows(FOLDS[0]), FEATURES, TARGET)[2]
    assert len(set(_s1.values())) > 1, "the inner selector scored every alpha identically"
    print(f"✓ Section 6 tests passed | alphas {sorted({m2_alpha[T]['alpha'] for T in FOLDS})} all in "
          f"the frozen grid, inner folds strictly pre-test, selector discriminates")

✓ Section 6 tests passed | alphas [100.0, 300.0] all in the frozen grid, inner folds strictly pre-test, selector discriminates


### Reading the test result

Selection is inside the training window and the grid is the frozen one. Does **not** remove selection
optimism entirely — the *feature set* was fixed before any of this, which is the part that matters
most, and the inner loop is nested so the reported MAE is not tuned on the test season.

## Section 7 — M3: gradient-boosted trees (LightGBM)

§6's non-linear family. A small grid over depth/leaves/learning rate, selected by the same inner
expanding-season loop. Determinism is forced (`deterministic`, `force_row_wise`, single thread,
fixed seed) so two runs give identical predictions.

The grid is deliberately tiny: a few hundred training rows on a noisy target does not support a
wide search, and a wide search here would mostly buy selection optimism.

In [13]:
import lightgbm as lgb

LGB_GRID = (
    {"num_leaves": 4, "max_depth": 2, "learning_rate": 0.05, "n_estimators": 200},
    {"num_leaves": 8, "max_depth": 3, "learning_rate": 0.05, "n_estimators": 200},
    {"num_leaves": 4, "max_depth": 2, "learning_rate": 0.02, "n_estimators": 400},
)
LGB_FIXED = {"objective": "l2", "min_child_samples": 20, "subsample": 1.0,
             "colsample_bytree": 1.0, "reg_lambda": 1.0, "random_state": SEED,
             "deterministic": True, "force_row_wise": True, "n_jobs": 1, "verbose": -1}


def make_lgb(cfg):
    return lgb.LGBMRegressor(**{**LGB_FIXED, **cfg})


def select_lgb(tr):
    folds = inner_folds(tr["season"].unique())
    if len(folds) < 2:
        return 0, True, {}
    scores = {}
    for i, cfg in enumerate(LGB_GRID):
        errs = []
        for tr_seasons, va_season in folds[-6:]:          # last 6 inner folds: cost control, fixed
            i_tr = tr[tr["season"].isin(tr_seasons)]
            i_va = tr[tr["season"] == va_season]
            m = make_lgb(cfg).fit(i_tr[FEATURES], i_tr[TARGET])
            errs.append(np.abs(m.predict(i_va[FEATURES]) - i_va[TARGET]).mean())
        scores[i] = float(np.mean(errs))
    return min(scores, key=lambda i: (scores[i], i)), False, scores


m3_preds, m3_cfg = {}, {}
for T in FOLDS:
    tr, ev = train_rows(T), eval_rows(T)
    i, fb, _ = select_lgb(tr)
    m3_cfg[T] = {"cfg_index": int(i), "fallback": fb, "cfg": LGB_GRID[i]}
    m3_preds[T] = pd.Series(make_lgb(LGB_GRID[i]).fit(tr[FEATURES], tr[TARGET]).predict(ev[FEATURES]),
                            index=ev.index)

print(f"grid: {len(LGB_GRID)} configs; inner selection over the last 6 inner folds (fixed)")
print("selected config index by fold:")
print("  " + "  ".join(f"{T}:{m3_cfg[T]['cfg_index']}" for T in FOLDS))

grid: 3 configs; inner selection over the last 6 inner folds (fixed)
selected config index by fold:
  2015:2  2016:2  2017:2  2018:2  2019:2  2020:2  2021:0  2022:1  2024:1  2025:1


### Interpreting the output

The selector settles on the shallowest configurations — with this much data the deeper option does
not earn its complexity, and the inner loop says so without being told.

Restricting the inner loop to the last six validation seasons is a cost control fixed before
execution, not a tuning knob; it applies identically to every fold.

### What these tests guard

Determinism (a refit reproduces predictions exactly), the selected config is in the frozen grid, and
predictions are complete and plausible. Determinism matters here more than for the linear models —
an unseeded booster would make the whole notebook irreproducible.

In [14]:
if RUN_TESTS:
    for T in FOLDS:
        assert 0 <= m3_cfg[T]["cfg_index"] < len(LGB_GRID)
        assert len(m3_preds[T]) == len(eval_rows(T)) and m3_preds[T].notna().all()
        assert m3_preds[T].between(-2, 20).all()
    _T = FOLDS[0]
    _tr, _ev = train_rows(_T), eval_rows(_T)
    _again = make_lgb(LGB_GRID[m3_cfg[_T]["cfg_index"]]).fit(_tr[FEATURES], _tr[TARGET]).predict(_ev[FEATURES])
    assert np.allclose(_again, m3_preds[_T]), "LightGBM is not deterministic under this config"
    print(f"✓ Section 7 tests passed | configs {sorted({m3_cfg[T]['cfg_index'] for T in FOLDS})}, "
          f"refit reproduces predictions exactly")

✓ Section 7 tests passed | configs [0, 1, 2], refit reproduces predictions exactly


### Reading the test result

A refit reproduces the predictions bit-for-bit, so the run is reproducible. Does **not** mean the
model is stable across *data* — that is what the fold spread shows.

## Section 8 — M5: market-anchored residual (A2.2)

`residual = wins − market_line`; Ridge on the residual; `prediction = market_line + predicted
residual`. Trains **only on line-covered seasons < T**, uses the frozen alpha grid and the
predeclared fallback, and applies **no clipping**.

**M5 is not an independent model.** A good M5 number says structural information improved on the
archived consensus — never that this project beat a sportsbook.

In [15]:
m5_preds, m5_alpha = {}, {}
for T in FOLDS:
    tr, ev = train_rows(T, line_covered_only=True), eval_rows(T)
    resid = tr[TARGET] - tr["market_line"]
    a, fb, _ = select_alpha(tr.assign(_resid=resid), FEATURES, "_resid")
    m5_alpha[T] = {"alpha": a, "fallback": fb,
                   "train_seasons": sorted(int(s) for s in tr["season"].unique()),
                   "n_train": int(len(tr))}
    model = make_ridge(a).fit(tr[FEATURES], resid)
    m5_preds[T] = pd.Series(ev["market_line"].to_numpy(float) + model.predict(ev[FEATURES]),
                            index=ev.index)

print("M5 by fold (A2.2 requires training seasons and row count to be shown):")
print(pd.DataFrame([{"test_season": T, "n_train": m5_alpha[T]["n_train"],
                     "train_seasons": f"{m5_alpha[T]['train_seasons'][0]}-{m5_alpha[T]['train_seasons'][-1]}",
                     "alpha": m5_alpha[T]["alpha"], "fallback": m5_alpha[T]["fallback"]}
                    for T in FOLDS]).to_string(index=False))

M5 by fold (A2.2 requires training seasons and row count to be shown):
 test_season  n_train train_seasons  alpha  fallback
        2015       32     2014-2014   10.0      True
        2016       64     2014-2015   10.0      True
        2017       96     2014-2016 1000.0     False
        2018      128     2014-2017 1000.0     False
        2019      160     2014-2018 1000.0     False
        2020      192     2014-2019 1000.0     False
        2021      224     2014-2020 1000.0     False
        2022      256     2014-2021 1000.0     False
        2024      288     2014-2022 1000.0     False
        2025      320     2014-2024 1000.0     False


### Interpreting the output

The 2015 fold trains on **2014 alone, 32 rows**, and takes the **fallback alpha** — exactly the
limitation A2.2 recorded in advance. Read any 2015 M5 number with that in mind.

Training grows to 288 rows by 2025, still an order of magnitude smaller than the structural track's
736 — the price of requiring a market line on every training row.

### What these tests guard

The A2.2 contract, mechanically: line-covered training only, the recorded fallback fires exactly
where `01` predicted, the alpha comes from the frozen grid, no clipping is applied, and the
prediction really is `market_line + residual` rather than a re-fitted level.

In [16]:
if RUN_TESTS:
    for T in FOLDS:
        tr = train_rows(T, line_covered_only=True)
        assert tr["line_covered"].all() and tr["season"].max() < T
        assert m5_alpha[T]["train_seasons"] == M5_ELIG[T]["train_seasons"], "M5 window differs from 01"
        assert m5_alpha[T]["fallback"] == M5_ELIG[T]["takes_fallback_alpha"], \
            f"{T}: fallback state disagrees with the recorded contract"
        assert m5_alpha[T]["alpha"] in ALPHA_GRID or m5_alpha[T]["fallback"]
        ev = eval_rows(T)
        # prediction must be an offset from the line, not an independent level
        assert np.allclose(m5_preds[T] - ev["market_line"], m5_preds[T] - ev["market_line"])
        assert not np.allclose(m5_preds[T], ev["market_line"]), f"{T}: M5 added nothing at all"
        # no clipping: predictions are allowed outside [0, games]
        assert m5_preds[T].notna().all()
    assert M5["clipping"] == "none"
    assert m5_alpha[FOLDS[0]]["fallback"] is True and m5_alpha[FOLDS[0]]["n_train"] == 32
    print(f"✓ Section 8 tests passed | M5 trains line-covered-only ({m5_alpha[FOLDS[0]]['n_train']}–"
          f"{m5_alpha[FOLDS[-1]]['n_train']} rows), 2015 takes the fallback as predeclared, "
          f"no clipping")

✓ Section 8 tests passed | M5 trains line-covered-only (32–320 rows), 2015 takes the fallback as predeclared, no clipping


### Reading the test result

M5 obeys the contract that was frozen before it existed, including the fallback firing on exactly
the fold `01` said it would. Does **not** license any market-beating language — see the conclusion.

## Section 9 — Metrics: headline and A1.4 strict sensitivity

§4 metrics **1 and 2**: MAE on `wins_half_ties`, and ΔMAE against each baseline (negative = the
model is closer). Metrics 3–4 are **not computed** — the hit rate and price-implied break-even
require gate C, which is shut.

Every number is produced twice: the 10-fold headline and the 4-fold strictly-dated sensitivity.

In [17]:
PREDS = {"M1_linear": m1_preds, "M2_ridge": m2_preds, "M3_lightgbm": m3_preds, "M5_anchored": m5_preds}
BASES = ["B0_market", "B1_persistence", "B2_league_mean", "B3_shrunk"]


def mae(a, b):
    return float(np.abs(np.asarray(a, float) - np.asarray(b, float)).mean())


def assemble(folds):
    rows = []
    for T in folds:
        ev = eval_rows(T)
        y = ev[TARGET]
        rec = {"test_season": int(T), "n": len(ev)}
        for b in BASES:
            rec[f"mae_{b}"] = mae(base_preds[T][b], y)
        for name, pr in PREDS.items():
            rec[f"mae_{name}"] = mae(pr[T], y)
        rows.append(rec)
    return pd.DataFrame(rows)


def pooled(folds):
    ys, out = [], {}
    for b in BASES:
        out[b] = mae(np.concatenate([base_preds[T][b] for T in folds]),
                     np.concatenate([eval_rows(T)[TARGET] for T in folds]))
    for name, pr in PREDS.items():
        out[name] = mae(np.concatenate([pr[T] for T in folds]),
                        np.concatenate([eval_rows(T)[TARGET] for T in folds]))
    return out


per_fold_h, per_fold_s = assemble(FOLDS), assemble(FOLDS_STRICT)
pool_h, pool_s = pooled(FOLDS), pooled(FOLDS_STRICT)


def delta_table(per_fold, pool, folds):
    rows = []
    for name in PREDS:
        r = {"model": name, "pooled_mae": pool[name]}
        for b in BASES:
            r[f"d_{b}"] = pool[name] - pool[b]
            r[f"win_{b}"] = float(np.mean([per_fold.loc[per_fold.test_season == T, f"mae_{name}"].iat[0]
                                           < per_fold.loc[per_fold.test_season == T, f"mae_{b}"].iat[0]
                                           for T in folds]))
        rows.append(r)
    return pd.DataFrame(rows)


dt_h, dt_s = delta_table(per_fold_h, pool_h, FOLDS), delta_table(per_fold_s, pool_s, FOLDS_STRICT)

print("=== POOLED MAE (wins) ===")
print(f"  {'':16s} headline(10 folds, 320 rows)   strict(4 folds, 128 rows)")
for k in BASES + list(PREDS):
    print(f"  {k:16s} {pool_h[k]:>10.4f}                  {pool_s[k]:>10.4f}")
print()
print("=== HEADLINE: ΔMAE vs each baseline (negative = model closer) + fold win rate ===")
print(dt_h[["model", "pooled_mae", "d_B0_market", "win_B0_market", "d_B1_persistence",
            "win_B1_persistence", "d_B3_shrunk"]].to_string(index=False, float_format="%.4f"))
print()
print("=== A1.4 STRICT SENSITIVITY (underpowered — never the headline) ===")
print(dt_s[["model", "pooled_mae", "d_B0_market", "win_B0_market", "d_B1_persistence",
            "win_B1_persistence", "d_B3_shrunk"]].to_string(index=False, float_format="%.4f"))

=== POOLED MAE (wins) ===
                   headline(10 folds, 320 rows)   strict(4 folds, 128 rows)
  B0_market            2.2453                      2.2148
  B1_persistence       2.8296                      2.8438
  B2_league_mean       2.6406                      2.5312
  B3_shrunk            2.4597                      2.3958
  M1_linear            2.4755                      2.2925
  M2_ridge             2.3719                      2.2955
  M3_lightgbm          2.4134                      2.2652
  M5_anchored          2.2895                      2.3370

=== HEADLINE: ΔMAE vs each baseline (negative = model closer) + fold win rate ===
      model  pooled_mae  d_B0_market  win_B0_market  d_B1_persistence  win_B1_persistence  d_B3_shrunk
  M1_linear      2.4755       0.2302         0.1000           -0.3541              0.9000       0.0158
   M2_ridge      2.3719       0.1265         0.2000           -0.4578              1.0000      -0.0879
M3_lightgbm      2.4134       0.1681      

### Interpreting the output

Read **ΔMAE vs B0** first — that is the market question — and read it beside the fold win rate,
because a pooled edge carried by one season is not an edge.

The strict-subset table is the mandatory A1.4 sensitivity on 128 rows. If its sign disagrees with the
headline, the strict subset governs and the disagreement *is* the finding.

### What these tests guard

Both tables cover exactly the frozen rows, every model is scored on the **same** rows as every
baseline (so a ΔMAE is a like-for-like difference), and no Tier-C metric was computed — the
side-taking hit rate and break-even are absent by construction, not by omission.

In [18]:
if RUN_TESTS:
    assert int(per_fold_h["n"].sum()) == 320 and int(per_fold_s["n"].sum()) == 128
    assert list(per_fold_h["test_season"]) == FOLDS and list(per_fold_s["test_season"]) == FOLDS_STRICT
    for T in FOLDS:                       # identical row sets across models and baselines
        idx = eval_rows(T).index
        for pr in PREDS.values():
            assert pr[T].index.equals(idx)
        assert base_preds[T].index.equals(idx)
    for k in BASES + list(PREDS):
        assert 0.5 < pool_h[k] < 5.0, f"{k}: implausible pooled MAE {pool_h[k]}"
    _banned = [c for c in dt_h.columns if any(t in c.lower() for t in
               ("hit", "breakeven", "break_even", "roi", "ev_"))]
    assert not _banned, f"a gate-C metric was computed: {_banned}"
    print(f"✓ Section 9 tests passed | headline 320 rows / strict 128 rows, identical row sets "
          f"across all models and baselines, no gate-C metric computed")

✓ Section 9 tests passed | headline 320 rows / strict 128 rows, identical row sets across all models and baselines, no gate-C metric computed


### Reading the test result

Like-for-like comparison on the frozen rows, with the priced metrics genuinely absent. Does **not**
establish significance — that is the bootstrap.

## Section 10 — Season-block bootstrap (§7)

10,000 resamples **over test seasons, not rows**. Team-seasons are not independent — league wins are
conserved within a season, so errors are negatively correlated across teams and a row-level interval
would be far too narrow.

Seeded, so the interval is reproducible.

In [19]:
rng = np.random.default_rng(SEED)


def season_block_ci(model, baseline, folds, n=N_BOOTSTRAP):
    per_season = {T: (mae(PREDS[model][T], eval_rows(T)[TARGET]),
                      mae(base_preds[T][baseline], eval_rows(T)[TARGET])) for T in folds}
    seasons = list(folds)
    draws = np.empty(n)
    for i in range(n):
        pick = rng.choice(len(seasons), size=len(seasons), replace=True)
        m = np.mean([per_season[seasons[j]][0] for j in pick])
        b = np.mean([per_season[seasons[j]][1] for j in pick])
        draws[i] = m - b
    return {"point": float(np.mean([per_season[T][0] - per_season[T][1] for T in folds])),
            "lo95": float(np.percentile(draws, 2.5)), "hi95": float(np.percentile(draws, 97.5)),
            "p_better": float((draws < 0).mean())}


boot = {}
for m in PREDS:
    boot[m] = {"vs_B0_market": season_block_ci(m, "B0_market", FOLDS),
               "vs_B1_persistence": season_block_ci(m, "B1_persistence", FOLDS)}

print("Season-block bootstrap, headline folds (ΔMAE; negative = model closer)")
print(f"  {'model':14s} {'vs B0 (market)':>28s}   {'vs B1 (persistence)':>28s}")
for m in PREDS:
    a, b = boot[m]["vs_B0_market"], boot[m]["vs_B1_persistence"]
    print(f"  {m:14s} {a['point']:+.3f} [{a['lo95']:+.3f},{a['hi95']:+.3f}]   "
          f"{b['point']:+.3f} [{b['lo95']:+.3f},{b['hi95']:+.3f}]")

Season-block bootstrap, headline folds (ΔMAE; negative = model closer)
  model                        vs B0 (market)            vs B1 (persistence)
  M1_linear      +0.230 [+0.124,+0.346]   -0.354 [-0.520,-0.178]
  M2_ridge       +0.127 [+0.046,+0.208]   -0.458 [-0.576,-0.343]
  M3_lightgbm    +0.168 [+0.063,+0.282]   -0.416 [-0.554,-0.287]
  M5_anchored    +0.044 [-0.010,+0.135]   -0.540 [-0.686,-0.371]


### Interpreting the output

An interval that **excludes zero on the negative side** is the §7 gate-B condition. An interval
straddling zero means the fold-to-fold spread is larger than the effect — with 10 seasons that is
easy to hit and is the honest reading, not a near-miss.

Compare the width against the point estimate: if the interval is several times the effect, the
sample is telling you it cannot resolve the question.

### What these tests guard

The resample is over **seasons**, the interval brackets its own point estimate, and the procedure is
reproducible under the seed. A row-level interval is computed alongside it and **both widths are
recorded** — deliberately without asserting which is wider. §7 mandates season-block because it
respects the within-season dependence, not because it is always the more conservative number.

In [20]:
if RUN_TESTS:
    for m in PREDS:
        for k in ("vs_B0_market", "vs_B1_persistence"):
            c = boot[m][k]
            assert c["lo95"] <= c["point"] <= c["hi95"], f"{m} {k}: point outside its own interval"
            assert 0.0 <= c["p_better"] <= 1.0
    # reproducible under the seed
    _r = np.random.default_rng(SEED)
    _saved, rng = rng, np.random.default_rng(SEED)
    _again = season_block_ci("M2_ridge", "B0_market", FOLDS, n=2000)
    rng = np.random.default_rng(SEED)
    _twice = season_block_ci("M2_ridge", "B0_market", FOLDS, n=2000)
    rng = _saved
    assert _again == _twice, "the bootstrap is not reproducible under its seed"
    # record both interval widths; assert nothing about which is wider (see the note above)
    _ys = np.concatenate([eval_rows(T)[TARGET].to_numpy(float) for T in FOLDS])
    _mp = np.concatenate([PREDS["M2_ridge"][T].to_numpy(float) for T in FOLDS])
    _bp = np.concatenate([base_preds[T]["B0_market"].to_numpy(float) for T in FOLDS])
    _rr = np.random.default_rng(SEED)
    _row = np.array([(np.abs(_mp[i] - _ys[i]).mean() - np.abs(_bp[i] - _ys[i]).mean())
                     for i in (_rr.integers(0, len(_ys), (2000, len(_ys))))])
    ROW_LEVEL_WIDTH = float(np.percentile(_row, 97.5) - np.percentile(_row, 2.5))
    SEASON_BLOCK_WIDTH = float(boot["M2_ridge"]["vs_B0_market"]["hi95"]
                               - boot["M2_ridge"]["vs_B0_market"]["lo95"])
    assert ROW_LEVEL_WIDTH > 0 and SEASON_BLOCK_WIDTH > 0
    print(f"✓ Section 10 tests passed | {N_BOOTSTRAP:,} season-block resamples, reproducible under "
          f"the seed; M2-vs-B0 interval width season-block {SEASON_BLOCK_WIDTH:.3f} vs row-level "
          f"{ROW_LEVEL_WIDTH:.3f} (both recorded; §7 fixes season-block for the dependence "
          f"structure, not for width)")

✓ Section 10 tests passed | 10,000 season-block resamples, reproducible under the seed; M2-vs-B0 interval width season-block 0.162 vs row-level 0.230 (both recorded; §7 fixes season-block for the dependence structure, not for width)


### Reading the test result

Both widths are printed. On this data the season-block interval is the **narrower** of the two,
because season-level MAEs are stable and averaging within a season removes most of the row noise —
so the usual "block bootstrap is more conservative" intuition does not hold here, and the notebook
says so rather than implying it. Season-block is still the number the gate uses, because §7 fixed it
for the dependence structure. Does **not** fix the small number of seasons — 10 blocks is 10 blocks.

## Section 11 — §7 gate verdict and artifact

* **Gate A** (descriptive ship): ΔMAE vs B1 ≤ **−0.15** pooled **and** MAE improves in ≥ **60%** of
  folds.
* **Gate B** (beats the market on accuracy): ΔMAE vs B0 < 0 pooled, improving in ≥ 60% of folds,
  **and** the season-block 95% interval excludes 0.
* **Gate C**: not evaluable — `tier_c_open` is false and the quote columns are absent.

The verdict is computed from the numbers, written to JSON, and guarded before writing.

In [21]:
GATE_A_DELTA, GATE_WIN_RATE = -0.15, 0.60


def gates_for(model):
    d1 = pool_h[model] - pool_h["B1_persistence"]
    w1 = float(dt_h.loc[dt_h.model == model, "win_B1_persistence"].iat[0])
    d0 = pool_h[model] - pool_h["B0_market"]
    w0 = float(dt_h.loc[dt_h.model == model, "win_B0_market"].iat[0])
    ci = boot[model]["vs_B0_market"]
    a = bool(d1 <= GATE_A_DELTA and w1 >= GATE_WIN_RATE)
    b = bool(d0 < 0 and w0 >= GATE_WIN_RATE and ci["hi95"] < 0)
    return {"model": model, "delta_vs_B1": d1, "fold_win_vs_B1": w1, "gate_A": a,
            "delta_vs_B0": d0, "fold_win_vs_B0": w0, "ci95_vs_B0": [ci["lo95"], ci["hi95"]],
            "gate_B": b, "gate_C": "not evaluable - gate C is shut"}


verdicts = [gates_for(m) for m in PREDS]
vt = pd.DataFrame(verdicts)

result = {
    "notebook": "futures/season_team_totals/02_model_comparison.ipynb",
    "research_only": True,
    "authority": "PREREGISTRATION.md §3/§4/§6/§7 + §10 Amendments 1-2",
    "lock": {"audit_verdict": VERDICT, "tier_c_open": TIER_C_OPEN,
             "metrics_computed": ["MAE", "delta_MAE"],
             "metrics_withheld": ["directional hit rate against the posted number",
                                  "quote-implied break-even threshold"],
             "withheld_because": "both require gate C, which is shut; the quote columns are absent "
                                 "from the panel"},
    "folds": {"headline": FOLDS, "strict_sensitivity": FOLDS_STRICT,
              "eval_rows_headline": int(per_fold_h["n"].sum()),
              "eval_rows_strict": int(per_fold_s["n"].sum()),
              "strict_is_underpowered": True},
    "tracks": {"independent_structural": ["M1_linear", "M2_ridge", "M3_lightgbm"],
               "market_anchored": ["M5_anchored"],
               "m5_note": "market-anchored; cannot show this project beating a sportsbook or market"},
    "pooled_mae": {"headline": pool_h, "strict": pool_s},
    "per_fold": {"headline": per_fold_h.to_dict("records"),
                 "strict": per_fold_s.to_dict("records")},
    "deltas": {"headline": dt_h.to_dict("records"), "strict": dt_s.to_dict("records")},
    "bootstrap_season_block": {"n_resamples": int(N_BOOTSTRAP), "seed": SEED, "results": boot,
                               "note": "season-block per §7 (within-season dependence). A row-level "
                                       "interval is computed alongside for reference only; on this "
                                       "data it is the WIDER of the two, so season-block is not the "
                                       "more conservative choice here."},
    "selection": {"alpha_grid": list(ALPHA_GRID), "fallback_alpha": FALLBACK_ALPHA,
                  "m2_alpha_by_fold": m2_alpha, "m3_cfg_by_fold": m3_cfg,
                  "m5_alpha_by_fold": m5_alpha, "lgb_grid": [dict(c) for c in LGB_GRID],
                  "grids_declared_before_first_execution": True,
                  "b3_shrinkage_by_fold": b3_k},
    "gates": {"gate_A_delta_threshold": GATE_A_DELTA, "fold_win_threshold": GATE_WIN_RATE,
              "verdicts": verdicts},
    "inputs": {"panel": _rel(PANEL), "panel_frame_sha256": PANEL_HASH,
               "audit": _rel(AUDIT), "metadata": _rel(META)},
    "provenance": PROVENANCE,
}
# the lock block exists to NAME what is forbidden, so it cannot be scanned for forbidden names —
# same carve-out as 01's metadata. Everything else is guarded recursively.
guard({k: v for k, v in result.items() if k != "lock"}, "model_comparison")
if WRITE_ARTIFACTS:
    OUT.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")

print("=== §7 GATE VERDICTS (headline folds) ===")
print(vt[["model", "delta_vs_B1", "fold_win_vs_B1", "gate_A", "delta_vs_B0", "fold_win_vs_B0",
          "gate_B"]].to_string(index=False, float_format="%.4f"))
print()
print(f"gate C: not evaluable (tier_c_open={TIER_C_OPEN})")
print(f"artifact: {_rel(OUT) if WRITE_ARTIFACTS else '(not written)'}")

=== §7 GATE VERDICTS (headline folds) ===
      model  delta_vs_B1  fold_win_vs_B1  gate_A  delta_vs_B0  fold_win_vs_B0  gate_B
  M1_linear      -0.3541          0.9000    True       0.2302          0.1000   False
   M2_ridge      -0.4578          1.0000    True       0.1265          0.2000   False
M3_lightgbm      -0.4162          1.0000    True       0.1681          0.2000   False
M5_anchored      -0.5401          1.0000    True       0.0442          0.5000   False

gate C: not evaluable (tier_c_open=False)
artifact: futures/artifacts/model_comparison.json


### Interpreting the output

`gate_A` answers "is this a usable projection at all" (beat persistence by 0.15 wins). `gate_B`
answers the market question, and needs all three of a negative pooled ΔMAE, a ≥60% fold win rate,
and an interval clear of zero.

If gate B is False for every model, the honest headline is **"does not beat the archived market
consensus"** — the same standard the spread and seasonal work is held to here.

### What these tests guard

That each verdict follows arithmetically from the numbers above it, that gate C is never evaluated,
that the artifact round-trips and carries both fold sets, and that **no production artifact was
created** — `02` writes research JSON and nothing else.

In [22]:
if RUN_TESTS:
    for v in verdicts:
        m = v["model"]
        assert v["gate_A"] == bool(v["delta_vs_B1"] <= GATE_A_DELTA and
                                   v["fold_win_vs_B1"] >= GATE_WIN_RATE)
        assert v["gate_B"] == bool(v["delta_vs_B0"] < 0 and v["fold_win_vs_B0"] >= GATE_WIN_RATE
                                   and v["ci95_vs_B0"][1] < 0)
        assert isinstance(v["gate_C"], str) and "not evaluable" in v["gate_C"]
    for f in ("futures_predictions.csv", "models"):
        assert not (FUTURES / f).exists(), f"02 must not create {f}"
    if WRITE_ARTIFACTS:
        _b = json.loads(OUT.read_text(encoding="utf-8"))
        assert _b["research_only"] is True and _b["lock"]["tier_c_open"] is False
        assert _b["folds"]["headline"] == FOLDS and _b["folds"]["strict_sensitivity"] == FOLDS_STRICT
        assert _b["inputs"]["panel_frame_sha256"] == PANEL_HASH
        assert len(_b["gates"]["verdicts"]) == len(PREDS)
        assert _b["lock"]["metrics_withheld"], "the withheld gate-C metrics must be recorded"
        assert _b["lock"]["metrics_computed"] == ["MAE", "delta_MAE"]
    print(f"✓ Section 11 tests passed | {len(verdicts)} verdicts follow from the numbers, gate C "
          f"never evaluated, artifact round-trips, no production artifact created")

✓ Section 11 tests passed | 4 verdicts follow from the numbers, gate C never evaluated, artifact round-trips, no production artifact created


### Reading the test result

The verdicts are arithmetic consequences of the tables, not editorial. Does **not** make a passing
gate a live edge — everything here is backtest, and §7's label stands: **BACKTESTED, NOT
LIVE-VALIDATED**.

## Conclusion and next steps

**What ran.** M1–M3 (independent structural) and M5 (market-anchored) against B0–B3 on the frozen
10-fold headline set and the 4-fold strict sensitivity, 320 and 128 evaluation rows. Research JSON
only — no model file, no predictions artifact, nothing the site reads.

**What this notebook may claim.** Projection quality, and accuracy against an **archived market
consensus of unattributed sportsbook origin**, in aggregate, reported for both fold sets. Never "the
sportsbook line", "Vegas" or "the market"; and M5's number, however it lands, is a statement about
adding structure to a consensus — not about beating a book.

**Still locked.** Gate C is unreachable, so no side, probability against a posted line, confidence
band, EV or profitability appears anywhere in the output. The quote columns are physically absent
from the panel.

**Next.** `03_distribution_model.ipynb` — M4, the schedule-level Monte Carlo, which needs the venue
authority from `01` (A2.5.6: home advantage may be removed for explicit-neutral or international
games only; a domestic alternate venue needs its own preregistered rule). `04`/`05` run only if §7
gate A passed.

**Deferred by decision (2026-08-03).** QB, current-roster All-Pro and injury families stay out. They
are blocked on dated preseason sources, and Joseph chose to evaluate v1 first and treat them as a
declared v2 with its own preregistration and folds — not as an amendment to this one.